# Parallel Requests Demo with Training, Embedding, and Search

This notebook demonstrates how to:
1. Read a text file (`cosmos.txt`), split it into sentences.
2. Use the first 10% of sentences to call the `/train` endpoint.
3. Use the remaining 90% of sentences to call the `/embed` endpoint in parallel.
4. Generate 20 questions about *Cosmos* by Carl Sagan and send them in parallel to the `/search` endpoint.

We use Python's `concurrent.futures` and `requests` to send multiple requests concurrently.

In [ ]:
# Install necessary libraries if not already installed
!pip install requests nltk

# Download nltk punkt tokenizer data
import nltk
nltk.download('punkt')
# Download the specific punkt_tab resource needed for sentence tokenization
nltk.download('punkt_tab')

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import nltk
from nltk.tokenize import sent_tokenize

# API endpoint URLs
TRAIN_URL = 'http://localhost:8000/train'
EMBED_URL = 'http://localhost:8000/embed'
SEARCH_URL = 'http://localhost:8000/search'

# Path to the text file containing content of 'Cosmos' by Carl Sagan
TEXT_FILE = 'cosmos.txt'


In [ ]:
# Read the book text from 'cosmos.txt'
with open(TEXT_FILE, 'r', encoding='utf-8') as f:
    text = f.read()

# Use nltk's sent_tokenize to split into sentences
sentences = sent_tokenize(text)
print(f"Total sentences extracted: {len(sentences)}")


In [ ]:
# Determine split index for 10% train, 95% embed
split_idx = max(1, int(len(sentences) * 0.1))  # Ensure at least one sentence in train
train_sentences = sentences[:split_idx]
embed_sentences = sentences[split_idx:]
print(f"Using {len(train_sentences)} sentences for training and {len(embed_sentences)} for embedding.")


In [ ]:
def call_train(sent_list):
    payload = {"sentences": sent_list}
    try:
        response = requests.post(TRAIN_URL, json=payload)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def call_embed(sent_list):
    payload = {"sentences": sent_list}
    try:
        response = requests.post(EMBED_URL, json=payload)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def call_search(query, k=5):
    payload = {"query": query, "k": k}
    try:
        response = requests.post(SEARCH_URL, json=payload)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}


In [ ]:
print("Calling /train endpoint...")
train_resp = call_train(train_sentences)
print("Train response:", train_resp)


In [ ]:
results_embed = []
print("Calling /embed endpoint in parallel for embed sentences...")
with ThreadPoolExecutor(max_workers=5) as executor:
    future_to_sent = {executor.submit(call_embed, [sent]): sent for sent in embed_sentences}
    for future in as_completed(future_to_sent):
        sent = future_to_sent[future]
        data = future.result()
        results_embed.append((sent, data))
print(f"Completed embedding {len(results_embed)} sentences.")
print(f"Sample result: {results_embed[0]}")


Below are 20 example questions about *Cosmos* by Carl Sagan. We will send these to the `/search` endpoint in parallel to retrieve nearest neighbors from the index.

In [ ]:
questions = [
    "What inspired Carl Sagan to write Cosmos?",
    "How does Cosmos explain the formation of galaxies?",
    "Describe the role of Carl Sagan in popularizing science.",
    "What is the significance of the Pale Blue Dot in Cosmos?",
    "How does Cosmos discuss the concept of light travel time?",
    "Explain Sagan's perspective on the search for extraterrestrial life.",
    "What does Cosmos say about the cosmic calendar?",
    "How are black holes described in Cosmos?",
    "What examples of cosmic evolution are provided in Cosmos?",
    "How does Sagan illustrate the scale of the universe?",
    "What role do molecules play according to Cosmos?",
    "Describe the segment on planetary exploration in Cosmos.",
    "What is Carl Sagan's view on the future of humanity?",
    "How does Cosmos address the origin of life?",
    "Explain the trilogy of Cosmos episodes related to stars.",
    "What narrative does Cosmos give about ancient civilizations?",
    "How does Sagan describe the importance of skepticism?",
    "What warnings does Cosmos offer regarding environmental issues?",
    "How are supernovas explained in Cosmos?",
    "What is the importance of the cosmic perspective in Cosmos?"
]
len(questions)


In [ ]:
results_search = []
print("Calling /search endpoint in parallel for questions...")
with ThreadPoolExecutor(max_workers=5) as executor:
    future_to_q = {executor.submit(call_search, q, 5): q for q in questions}
    for future in as_completed(future_to_q):
        q = future_to_q[future]
        data = future.result()
        results_search.append((q, data))
print(f"Completed search for {len(results_search)} questions.")

# Print search results
for idx, (q, res) in enumerate(results_search, 1):
    print(f"Question {idx}: {q}")
    print(res)
    print('-' * 60)
